In [ ]:
# To make sure the created documents are at hte project root level
import os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
print(os.getcwd())

In [ ]:
# Make sure any update to imported module reloads automatically
%load_ext autoreload
%autoreload 2

In [ ]:
# Fetch and store data
from qde.storage import save_ohlcv
save_ohlcv("BTCUSDT", source="binance", start="2015-01-01")
save_ohlcv("SPY", source="yfinance", start="2015-01-01")


In [ ]:
# Load stored data (no API call, instant)
from qde.storage import load_ohlcv_local

print("about to load")
df = load_ohlcv_local("BTCUSDT", source="binance")
print(f"loaded: {len(df)} rows")
print(df.head())

In [ ]:
# Query with SQL via DuckDB
from qde.storage import query

df = query("SELECT date, close FROM BTCUSDT_binance_1d WHERE close > 60000")
print(df.head())

In [ ]:
# Update with only new data
from qde.storage import update_ohlcv

update_ohlcv("BTCUSDT", source="binance")

In [ ]:
from qde.storage import query
print(query("SELECT count(*) as rows FROM BTCUSDT_binance_1d"))
print(query("SELECT count(*) as rows FROM SPY_yfinance_1d"))

In [ ]:
# Check the state of the btc data in storage
from qde.storage import load_ohlcv_local

df = load_ohlcv_local('BTCUSDT', source='binance')
print(f'Rows: {len(df)}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(df.head(10))
print(df.tail(10))
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
# Check the state of the spy data in storage
from qde.storage import load_ohlcv_local

df = load_ohlcv_local('SPY', source='yfinance')
print(f'Rows: {len(df)}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(df.head(10))
print(df.tail(10))
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
import pandas as pd
from qde.storage import load_ohlcv_local

df = load_ohlcv_local('BTCUSDT', source='binance')

full_range = pd.date_range(df.index.min(), df.index.max(), freq='D', tz='UTC')
missing = full_range.difference(df.index)
print(missing[:10])

In [ ]:
import pandas as pd
from qde.storage import load_ohlcv_local

df = load_ohlcv_local('SPY', source='yfinance')

full_range = pd.date_range(df.index.min(), df.index.max(), freq='D', tz='UTC')
missing = full_range.difference(df.index)
print(missing[:10].dayofweek)

In [ ]:
from qde.quality import check_gaps

btc = load_ohlcv_local("BTCUSDT", source="binance")
spy = load_ohlcv_local("SPY", source="yfinance")

print(f"BTC gaps: {len(check_gaps(btc, calendar='crypto'))}")
print(f"SPY gaps: {len(check_gaps(spy, calendar='equity'))}")

In [ ]:
df = load_ohlcv_local("BTCUSDT", source="binance")
print(df.index.duplicated().sum())
print(df[df.index.duplicated(keep=False)])

In [ ]:
df = load_ohlcv_local("SPY", source="yfinance")
print(df.index.duplicated().sum())
print(df[df.index.duplicated(keep=False)])

In [ ]:
from qde.quality import check_duplicates
spy = load_ohlcv_local("SPY", source="yfinance")
btc = load_ohlcv_local("BTCUSDT", source="binance")

btc_dupli = check_duplicates(btc)
spy_dupli = check_duplicates(spy)

print(btc_dupli)
print(spy_dupli)


In [ ]:
# Find null rows
df = load_ohlcv_local("BTCUSDT", source="binance")
print(df.isnull().sum())

In [ ]:

# Find null rows
df = load_ohlcv_local("SPY", source="yfinance")
print(df.isnull().sum())

In [ ]:
# Check for nulls
from qde.quality import check_nulls
spy = load_ohlcv_local("SPY", source="yfinance")
spy_null = check_nulls(spy)
print(spy_null)

In [ ]:
# Check for nulls
from qde.quality import check_nulls
btc = load_ohlcv_local("BTCUSDT", source="binance")
btc_null = check_nulls(btc)
print(btc_null)

In [ ]:
# Price sanity
from qde.storage import load_ohlcv_local
from qde.quality import check_price_sanity

from qde.quality import check_nulls
btc = load_ohlcv_local("BTCUSDT", source="binance")
spy = load_ohlcv_local("SPY", source="yfinance")

btc_sanity = check_price_sanity(btc)
spy_sanity = check_price_sanity(spy)

print(btc_sanity)
print(spy_sanity)



In [ ]:
# Status check

from qde.quality import run_quality_report
from qde.storage import load_ohlcv_local

btc = load_ohlcv_local("BTCUSDT", source="binance")
spy = load_ohlcv_local("SPY", source="yfinance")

run_quality_report(btc, name="BTCUSDT_binance_1d", calendar="crypto")
print()
run_quality_report(spy, name="SPY_yfinance_1d", calendar="equity")

In [ ]:
# Check nyse calendar for number if trading days
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date="2015-01-01", end_date="2026-06-18")
print(f'Trading days: {len(schedule)}')
print(schedule.head())

In [ ]:
# Check current data to compare the number of trading days

from qde.storage import load_ohlcv_local

spy = load_ohlcv_local("SPY", source="yfinance")
print(f'SPY rows: {len(spy)}')
print(f'NYSE trading days in range: {len(schedule)}')

In [ ]:
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date="2024-01-01", end_date="2024-02-01")
print(schedule.index.tz)

In [ ]:
from pathlib import Path

files =  list((Path("data") / "ohlcv").glob("*.parquet"))
for file in files:
    print(file.stem)

In [ ]:
# Split file name

name = "BTCUSDT_binance_1d"
parts = name.split("_")
print(parts)

In [ ]:
# Grab each part

symbol = parts[0]
source = parts[1]
interval = parts[2]

print(symbol, source, interval)

In [ ]:
# Load file and compute the stats
import pandas as pd

df = pd.read_parquet(files[0])
print(f"Rows: {len(df)}")
print(f"First: {df.index.min()}")
print(f"Last: {df.index.max()}")
print(f"Days stale: {(pd.Timestamp.now(tz='UTC') - df.index.max()).days}")

In [ ]:
# Run the quality checks and build one row

from qde.quality import check_gaps, check_nulls, check_duplicates, check_price_sanity

gaps = check_gaps(df, calendar="crypto")
duplicates = check_duplicates(df)
nulls = check_nulls(df)
price_issues = check_price_sanity(df)

row = {
    "symbol": symbol,
    "source": source,
    "interval": interval,
    "total_rows": len(df),
    "first_date": df.index.min(),
    "last_date": df.index.max(),
    "days_stale": (pd.Timestamp.now(tz="UTC") - df.index.max()).days,
    "gaps": len(gaps),
    "duplicates": len(duplicates),
    "nulls": nulls.sum(),
    "price_issues": len(price_issues),
}

row["status"] = "CLEAN" if (row["gaps"] == 0 and row["duplicates"] == 0 and row["nulls"] == 0 and row["price_issues"] == 0) else "ISSUES FOUND"

print(row)

In [ ]:
# Loop through all files and and collect a row for each

all_rows = []

for file in files:
    parts = file.stem.split("_")
    symbol = parts[0]
    source = parts[1]
    interval = parts[2]

    df = pd.read_parquet(file)

    # Determine calendar type
    calendar = "equity" if source == "yfinance" else "crypto"

    gaps = check_gaps(df, calendar=calendar)
    duplicates = check_duplicates(df)
    nulls = check_nulls(df)
    price_issues = check_price_sanity(df)

    row = {
        "symbol": symbol,
        "source": source,
        "interval": interval,
        "total_rows": len(df),
        "first_date": df.index.min(),
        "last_date": df.index.max(),
        "days_stale": (pd.Timestamp.now(tz="UTC") - df.index.max()).days,
        "gaps": len(gaps),
        "duplicates": len(duplicates),
        "nulls": nulls.sum(),
        "price_issues": len(price_issues),
    }
    row["status"] = "CLEAN" if (row["gaps"] == 0 and row["duplicates"] == 0 and row["nulls"] == 0 and row["price_issues"] == 0) else "ISSUES FOUND"

    all_rows.append(row)

summary = pd.DataFrame(all_rows)

summary.to_csv("data/quality_summary.csv", index=False)
summary



In [ ]:
from qde.quality import build_quality_summary

summary = build_quality_summary()
summary

In [ ]:
from qde.storage import load_ohlcv_local

spy = load_ohlcv_local("SPY", source="yfinance")
dupes = spy[spy.index.duplicated(keep=False)]
print(dupes)

In [ ]:
from qde.loaders import load_ohlcv

df = load_ohlcv("SPY", start="2026-06-18", source="yfinance")
print(df)

In [ ]:
from qde.storage import save_ohlcv

save_ohlcv("ETHUSDT", source="binance", start="2017-01-01")
# save_ohlcv("SOLUSDT", source="binance", start="2017-01-01")
save_ohlcv("GLD", source="yfinance", start="2015-01-01")
save_ohlcv("TLT", source="yfinance", start="2015-01-01")
save_ohlcv("QQQ", source="yfinance", start="2015-01-01")
save_ohlcv("TLT", source="yfinance", start="2015-01-01")
save_ohlcv("DX-Y.NYB", source="yfinance", start="2015-01-01")

